In many Q&A application we want to allow the user to have a back and forth coversation, meaning the application needs some "memory" of past questions and answers, and some logic for incorporating those into its current thinking .

Approches:


*   Chains, in which we always execute a retrieval step;
*   Agents, in which we give an LLM discretion over whether and how to execute a retrieval step (or multiple step).



In [2]:
!pip install langchain-groq langchain-classic langchain-huggingface langchain-chroma langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
Hf_token=os.getenv('Hf_Token')

from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key, model ="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x789bf8b5aae0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x789bf89800b0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Updated import for LangChain v1.0+
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


#load,chunk and index the content of the blog to create a retriever
import bs4
loader=WebBaseLoader(
    web_path=("https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/"),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)
docs=loader.load()
docs

[Document(metadata={'source': 'https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/'}, page_content='\nMost enterprise AI pilots fail for a reason that has nothing to do with the model.\nThe model works. The demo impresses. Then the project tries to scale across the organisation — and stalls.\nThe reason is structural. One AI assistant cannot realistically understand every department, every workflow, every data source, and every business rule inside a large enterprise. Finance behaves differently from supply chain. Compliance differs from customer operations. Internal knowledge is fragmented across CRMs, ERPs, tickets, PDFs, and legacy databases that nobody owns end to end.\nThis is where multi-agent AI changes the equation.\nInstead of one overloaded assistant trying to do everything, enterprises can deploy multiple specialised agents that collaborate. One retrieves data. Another validates policies. A third generates recommenda

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits=splitter.split_documents(docs)

In [8]:
len(splits)

14

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding=HuggingFaceEmbeddings(model="all-MiniLm-L6-v2")
embedding

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings(model_name='all-MiniLm-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [10]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(docs,embedding=embedding)
vectorstore

In [11]:
#querying with vectorstore
response=vectorstore.similarity_search_with_score("what did i asked you earlier ..?")
# response


In [12]:
retriever=vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x789bbaa59490>, search_kwargs={})

In [13]:
# Chatprompt template
from langchain_core.prompts import ChatPromptTemplate

system_prompt=(
        'you are an assistant for question-answering tasks.'
        "use the following pieces of retrieved context to answer"
        "the question.If you dont know the answer , say that you dont konw."
        "Use three sentence maximum and keep the answer concise."
        "{context}"
    )
prompt=ChatPromptTemplate.from_messages(

    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)



In [14]:
from langchain_classic.chains import create_retrieval_chain

question_answer=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer )
# response = question_answer.invoke({"input": "Hows you ?", "context": []})
# print(response)
response = rag_chain.invoke({"input": "what does cloudJournee do ..?"})
response

{'input': 'what does cloudJournee do ..?',
 'context': [Document(id='f48e96e9-e113-4cd6-a355-b6bdc297fcba', metadata={'source': 'https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/'}, page_content='\nMost enterprise AI pilots fail for a reason that has nothing to do with the model.\nThe model works. The demo impresses. Then the project tries to scale across the organisation — and stalls.\nThe reason is structural. One AI assistant cannot realistically understand every department, every workflow, every data source, and every business rule inside a large enterprise. Finance behaves differently from supply chain. Compliance differs from customer operations. Internal knowledge is fragmented across CRMs, ERPs, tickets, PDFs, and legacy databases that nobody owns end to end.\nThis is where multi-agent AI changes the equation.\nInstead of one overloaded assistant trying to do everything, enterprises can deploy multiple specialised age

In [12]:
response['answer']

'CloudJournee is an AWS Advanced Tier Partner with the AWS AI Competency. They design, build, and operate production-grade multi-agent AI systems on Amazon Bedrock for enterprise use cases.'

Flow :
User Question

    ↓
Retriever

      ↓
Relevant Documents

      ↓
Stuff into {context}

      ↓
LLM answers

In [13]:
# respon se=question_answer.invoke({'input':"what did i asked you earlier ","context":[]})
# print(response)

Adding ChatHistory for conversational QA

In [15]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# this is only the question rewriting prompt
contextual_system_prompt=(
    """
    Given a chat history and the latest user question,
    which migh reference context in the chat history,
    formulate a standalone wuestion which can be understood,
    without the user chat history , dont answer any question
    instead just reply with ,I Don't know , its wasnt found in
    conversationaal history
    """
)

contextual_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",contextual_system_prompt),

        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

history_aware_retriever=create_history_aware_retriever(llm,retriever,contextual_prompt )
# history_aware_retriever

In [16]:
history_aware_retriever.invoke({"chat_history":[],
            "input":"what is cloudjournee ?"})

[Document(id='f48e96e9-e113-4cd6-a355-b6bdc297fcba', metadata={'source': 'https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/'}, page_content='\nMost enterprise AI pilots fail for a reason that has nothing to do with the model.\nThe model works. The demo impresses. Then the project tries to scale across the organisation — and stalls.\nThe reason is structural. One AI assistant cannot realistically understand every department, every workflow, every data source, and every business rule inside a large enterprise. Finance behaves differently from supply chain. Compliance differs from customer operations. Internal knowledge is fragmented across CRMs, ERPs, tickets, PDFs, and legacy databases that nobody owns end to end.\nThis is where multi-agent AI changes the equation.\nInstead of one overloaded assistant trying to do everything, enterprises can deploy multiple specialised agents that collaborate. One retrieves data. Another valid

create_history_aware_retriever only USES the chat history temporarily.

chat_history

      ↓
history-aware retriever reads it

      ↓
rewrites question

      ↓
retriever searches better

In [17]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        MessagesPlaceholder('chat_history'),
        ("human","{input}")

    ]
)

question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain= create_retrieval_chain(history_aware_retriever,question_answer_chain)

from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
# question="what is CLoudJourness Mission and Vision..?"
question="what is CLoudJourness ..?"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})



In [18]:
print(response1['answer'])

CloudJournee is an AWS Advanced Tier Partner with the AWS AI Competency. They specialize in designing, building, and operating production-grade multi-agent AI systems on Amazon Bedrock for enterprise use cases.


In [19]:
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response1['answer'])
])
#the above part stores conversation


question2 = "tell me more about it ..?"
response2=rag_chain.invoke({"input":question2,"chat_history":chat_history})
print(response2['answer'])



I don't have more information about CloudJournee beyond what's mentioned in the provided context.


In [24]:
# from langchain_community.chat_message_histories import ChatMessageHistory
# from langchain_core.chat_history import BaseChatMessageHistory
# from langchain_core.runnables.history import RunnableWithMessageHistory

# store = {}

# def get_session_history(session_id:str)-> BaseChatMessageHistory:
#   if session_id not in store:
#     store[session_id] = ChatMessageHistory()
#   return store[session_id]


# conversational_rag_chain=RunnableWithMessageHistory(
#     rag_chain,
#     get_session_history,
#     input_message_key="input",
#     history_message_keys="chat_history",
#     output_messages_key="answer",
#     )

# conversational_rag_chain.invoke(
#     {"input":"what technologies does clousjourness uses?"},
#     config={
#         "configurable":{"session_id":"chat123"}
#         },
# )["answer"]


In [22]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        MessagesPlaceholder('chat_history'),
        ("human","{input}")

    ]
)

question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)

# Corrected rag_chain definition using LCEL to properly pass all inputs
from langchain_core.runnables import RunnablePassthrough # Added this import

rag_chain = (
    RunnablePassthrough.assign(
        context=history_aware_retriever, # `history_aware_retriever` expects input and chat_history
    ) # This step will output {'input': ..., 'chat_history': ..., 'context': [...]}
    | question_answer_chain # `question_answer_chain` expects context, input, and chat_history
)

from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
# question="what is CLoudJourness Mission and Vision..?"
question="what is CLoudJourness ..?"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})


In [23]:
response1

'CloudJournee is an AWS Advanced Tier Partner with the AWS AI Competency. They provide services for designing, building, and operating production-grade multi-agent AI systems on Amazon Bedrock for enterprise use cases.'